In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py

In [2]:
#Cargar datos
LABELS = ["Alt_Si3N4", "Anc_Si3N4", "lambda", "n_eff", "A_eff"]

def load_and_normalize(fname):
    with h5py.File(fname, "r") as f:
        A = f["/A"][...]
    axis5 = next((ax for ax, s in enumerate(A.shape) if s == 5), None)
    if axis5 is None:
        raise ValueError(f"No encontré eje de tamaño 5 en {fname}")
    if axis5 != A.ndim - 1:
        A = np.moveaxis(A, axis5, -1)
    return A

In [7]:
archivos = [
    "Datos_reto/Datos9.h5",
    "Datos_reto/Datos11.h5"
]
datasets = [load_and_normalize(fname) for fname in archivos]

def check_borders(d1, d2):
    alt1_end = np.round(d1[-1,0,0,0],6)
    alt2_start = np.round(d2[0,0,0,0],6)
    anc1_end = np.round(d1[0,-1,0,1],6)
    anc2_start = np.round(d2[0,0,0,1],6)
    print(f"Check Alt: {alt1_end} vs {alt2_start}, Check Anc: {anc1_end} vs {anc2_start}")
    if not np.isclose(alt1_end, alt2_start) and not np.isclose(anc1_end, anc2_start):
        print("⚠️ Advertencia: los bordes no coinciden perfectamente")

for i in range(len(datasets)-1):
    check_borders(datasets[i], datasets[i+1])

DatosU = np.concatenate(datasets, axis=0)

with h5py.File("Datos_reto/DatosU.h5", "w") as f_out:
    f_out.create_dataset("A", data=DatosU)

print("✅ Dataset unificado guardado en DatosU.h5, shape:", DatosU.shape)

def to_dataframe(arr):
    nA, nB, nC, _ = arr.shape
    rows = []
    for ii in range(nA):
        for jj in range(nB):
            for kk in range(nC):
                rec = {"i": ii+1, "j": jj+1, "k": kk+1}
                for p, name in enumerate(LABELS):
                    rec[name] = arr[ii, jj, kk, p]
                rows.append(rec)
    return pd.DataFrame(rows, columns=["i","j","k"]+LABELS)

df = to_dataframe(DatosU)
df["lambda_nm"] = df["lambda"] * 1000 
print(df)

Check Alt: 0.4 vs 0.4, Check Anc: 2.0 vs 0.5
✅ Dataset unificado guardado en DatosU.h5, shape: (100, 10, 10, 5)
        i   j   k  Alt_Si3N4  Anc_Si3N4  lambda     n_eff         A_eff  \
0       1   1   1   0.400000        0.5   1.035  1.594476  3.233126e-13   
1       1   1   2   0.466667        0.5   1.035  1.627080  3.328044e-13   
2       1   1   3   0.533333        0.5   1.035  1.652778  3.492393e-13   
3       1   1   4   0.600000        0.5   1.035  1.672627  3.684361e-13   
4       1   1   5   0.666667        0.5   1.035  1.688365  3.893655e-13   
...   ...  ..  ..        ...        ...     ...       ...           ...   
9995  100  10   6   0.733333        2.0   1.035  1.920578  9.078877e-13   
9996  100  10   7   0.800000        2.0   1.035  1.930293  9.668587e-13   
9997  100  10   8   0.866667        2.0   1.035  1.938219  1.026122e-12   
9998  100  10   9   0.933333        2.0   1.035  1.944864  1.086505e-12   
9999  100  10  10   1.000000        2.0   1.035  1.950377  1.14

In [10]:
c = 1      #Unidades naturales

pg = 30

intersecciones = np.array([])

def neff(lam, p):
    return np.polyval(p, lam)

def neff_p(lam, p):
    dp = np.polyder(p)
    return np.polyval(dp, lam)

def k(lam, p):
    return (2*np.pi/lam) * neff(lam, p)

def delta_k(lamp, lams, lami, p):
    return 2*k(lamp, p) - k(lams, p) - k(lami, p)

In [13]:
#intento con una geometria aleatoria hacer el grafico de ts = -ti

jp = 6           #Ancho
kp = 6           #Alto

df_p = df[(df['j']==jp) & (df['k']==kp)]

lam_p = df_p['lambda'].values
neff_p = df_p['n_eff'].values

alto = df_p['Alt_Si3N4'].iloc[0]
ancho = df_p['Anc_Si3N4'].iloc[0]

print(f"Geometría seleccionada: j={jp}, k={kp}")
print(f"Altura = {alto:.4f} µm")
print(f"Ancho  = {ancho:.4f} µm")

Geometría seleccionada: j=6, k=6
Altura = 0.7333 µm
Ancho  = 1.3333 µm


In [14]:
def neffvlambda(df, grado=30):
    if len(df) <= grado:
        raise ValueError("No hay suficientes puntos para ajuste polinomial.")

    x = df["lambda"].values
    y = df["n_eff"].values

    coef = np.polyfit(x, y, grado)
    p = np.poly1d(coef)
    return p    

In [21]:
#p = neffvlambda(df, pg)

lamp = np.linspace(0.5, 1.6, 200)
lams = np.linspace(0.5, 1.6, 200)
Lamp, Lams = np.meshgrid(lamp, lams)

Lami = 1/((2/Lamp)-(1/Lams))

mask = np.isfinite(Lami) & (Lami > 0)

print(Lami[mask])

[0.5        0.51117886 0.52261048 ... 1.57804117 1.58898279 1.6       ]
